# imports

In [2]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# config

In [1]:
ASSET      = "AAPL"
INTERVAL   = "1h"
HORIZON    = 4

# connection and load

In [3]:
db_path = '../../database/financial_data.duckdb'
conn = duckdb.connect(db_path)
df = conn.execute(f"""
    SELECT * FROM gold_ml_features
    WHERE asset_symbol = '{ASSET}'
    AND interval = '{INTERVAL}'
    ORDER BY date ASC
""").df()
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
print(f"Loaded {len(df)} rows for {ASSET} {INTERVAL}")

Loaded 3473 rows for AAPL 1h


# drop metadata cols

In [4]:
metadata_cols = ['asset_symbol', 'asset_class', 'exchange', 'interval']
df = df.drop(columns=metadata_cols, errors='ignore')
print(f"Columns remaining: {len(df.columns)}")

Columns remaining: 44


# 4hr target , return over 4 periods shift(-4)

In [5]:
df['target_4h'] = df['close'].pct_change(HORIZON).shift(-HORIZON)
df = df.dropna(subset=['target_4h'])
df['target_direction'] = (df['target_4h'] > 0).astype(int)
balance = df['target_direction'].value_counts(normalize=True) * 100
print("Class Balance (4h target):")
print(balance)
print(f"\nRows remaining after target creation: {len(df)}")

Class Balance (4h target):
target_direction
1    54.597867
0    45.402133
Name: proportion, dtype: float64

Rows remaining after target creation: 3469
